### Normal Dash

```python
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. CARGA Y PREPARACIÓN DE DATOS
# Cargamos el archivo CSV usando pandas
df = pd.read_csv("resources/gapminder.csv")

# Extraemos los años únicos y creamos una lista para el menú desplegable (dropdown)
# El formato debe ser: [{'label': '1952', 'value': 1952}, ...]
lista_años = sorted(df["year"].unique())
opciones_años = [{"label": str(y), "value": y} for y in lista_años]

# Definimos las columnas que se mostrarán en la tabla AG Grid
# 'field' debe coincidir exactamente con el nombre de la columna en el CSV
columnas_tabla = [
    {"field": "continent", "headerName": "Continente"},
    {"field": "country", "headerName": "País"},
    {"field": "year", "headerName": "Año"},
    {
        "field": "pop",
        "headerName": "Población",
        # TRUCO: Formato de miles con JS (separado por comas)
        "valueFormatter": {"function": "d3.format(',.0f')(params.value)"},
    },
    {
        "field": "gdpPercap",
        "headerName": "PIB per Cápita",
        # TRUCO: Formato de moneda
        "valueFormatter": {"function": "d3.format('$,.2f')(params.value)"},
    },
]
# 2. INICIALIZACIÓN DE LA APP
app = Dash(__name__)

# 3. DISEÑO DE LA INTERFAZ (LAYOUT)
# Aquí definimos cómo se ve la aplicación (títulos, botones, gráficos)
app.layout = html.Div(
    [
        html.H1("🌍 Explorador Gapminder", style={"textAlign": "center"}),
        # Fila de Selectores (Controles del usuario)
        html.Div(
            [
                # Selector de Métrica (Radio Items)
                html.Div(
                    [
                        html.Label(
                            "Selecciona una Métrica: ", style={"fontWeight": "bold"}
                        ),
                        dcc.RadioItems(
                            id="metric-radio",
                            options=[
                                {"label": "Población", "value": "pop"},
                                {"label": "Expectativa de Vida", "value": "lifeExp"},
                                {"label": "PIB per Cápita", "value": "gdpPercap"},
                            ],
                            value="lifeExp",  # Valor inicial por defecto
                            inline=True,
                        ),
                    ],
                    style={"flex": "1"},
                ),
                # Selector de Año (Dropdown)
                html.Div(
                    [
                        html.Label("Selecciona un Año: ", style={"fontWeight": "bold"}),
                        dcc.Dropdown(
                            id="year-dropdown",
                            options=opciones_años,
                            value=2007,  # Año inicial por defecto
                            clearable=False,
                        ),
                    ],
                    style={"flex": "1"},
                ),
            ],
            style={"display": "flex", "margin": "20px", "gap": "20px"},
        ),
        # Fila de Gráficos
        html.Div(
            [
                dcc.Graph(id="bar-chart", style={"flex": "1"}),
                dcc.Graph(id="geo-map", style={"flex": "1"}),
            ],
            style={"display": "flex", "gap": "10px"},
        ),
        # Sección de la Tabla
        html.H3("Tabla de Datos Crudos", style={"marginTop": "30px"}),
        dag.AgGrid(
            id="data-grid",
            columnDefs=columnas_tabla,
            # 'defaultColDef' aplica configuración (sort/filter) a todas las columnas
            defaultColDef={"sortable": True, "filter": True},
            # Habilitamos paginación para no saturar la vista
            dashGridOptions={"pagination": True},
        ),
    ],
    style={"padding": "20px", "fontFamily": "sans-serif"},
)


# 4. LÓGICA DE INTERACTIVIDAD (CALLBACK)
# Este bloque es el "cerebro" de la app: detecta cambios en los filtros y actualiza el UI
@callback(
    Output("bar-chart", "figure"),  # Actualiza el gráfico de barras
    Output("geo-map", "figure"),  # Actualiza el mapa
    Output("data-grid", "rowData"),  # Actualiza los datos de la tabla
    Input("metric-radio", "value"),  # Escucha el cambio de métrica
    Input("year-dropdown", "value"),  # Escucha el cambio de año
)
def actualizar_interfaz(metrica_seleccionada, año_seleccionado):
    """
    Esta función se ejecuta automáticamente cada vez que el usuario
    cambia un valor en los RadioItems o en el Dropdown.
    """

    # Filtrar el DataFrame original según el año elegido
    df_filtrado = df[df["year"] == año_seleccionado]

    # 1. Crear Gráfico de Barras: Promedio de la métrica por continente
    fig_barras = px.histogram(
        df_filtrado,
        x="continent",
        y=metrica_seleccionada,
        histfunc="avg",  # Calcula el promedio automáticamente
        color="continent",
        title=f"Promedio de {metrica_seleccionada} por Continente ({año_seleccionado})",
        labels={"continent": "Continente", metrica_seleccionada: "Valor Promedio"},
    )

    # 2. Crear Mapa Global de burbujas
    fig_mapa = px.scatter_geo(
        df_filtrado,
        locations="country",
        locationmode="country names",
        size=metrica_seleccionada,  # El tamaño de la burbuja depende del valor
        color="continent",
        hover_name="country",  # Muestra el nombre del país al pasar el mouse
        title=f"Distribución Global de {metrica_seleccionada} en {año_seleccionado}",
        # TRUCO: Quita los nombres de las columnas en el tooltip para que se vea más limpio
        hover_data={metrica_seleccionada: True, "continent": False, "country": False},
        # TRUCO: Renombra la métrica para que no aparezca el nombre técnico (ej: 'lifeExp')
        labels={metrica_seleccionada: "Valor Actual"},
    )

    # 3. Preparar datos para la tabla AG Grid
    # AG Grid requiere una lista de diccionarios (formato JSON)
    datos_tabla = df_filtrado.to_dict("records")

    # TRUCO: Antes de hacer el return en tu callback:
    fig_barras.update_layout(transition_duration=500)
    fig_mapa.update_layout(transition_duration=500)

    # Retornamos los tres elementos en el mismo orden que los 'Output' del decorador
    return fig_barras, fig_mapa, datos_tabla


# 5. EJECUCIÓN DE LA APLICACIÓN
if __name__ == "__main__":
    app.run(debug=True)
```

### Dash HTML and CSS

```python
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. Data Preparation
df = pd.read_csv("resources/gapminder.csv")
years = sorted(df["year"].unique())

# 2. Simplified Configs
cols = ["country", "year", "continent", "pop", "lifeExp", "gdpPercap"]
column_defs = [{"field": c} for c in cols]

app = Dash(__name__)

# 3. Layout using HTML Components and CSS Styles
app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": "#f4f4f9",
        "padding": "20px",
    },
    children=[
        # HTML Header Section
        html.Header(
            style={"textAlign": "center", "marginBottom": "30px"},
            children=[
                html.H1(
                    "Gapminder Data Explorer", style={"color": "#333", "margin": "0"}
                ),
                html.P("Analyze global trends through time", style={"color": "#666"}),
            ],
        ),
        # Controls Section (using CSS Flexbox for layout)
        html.Div(
            className="control-panel",  # You can define '.control-panel' in a separate assets/style.css
            style={
                "display": "flex",
                "gap": "20px",
                "backgroundColor": "white",
                "padding": "20px",
                "borderRadius": "8px",
                "boxShadow": "0 2px 4px rgba(0,0,0,0.1)",
                "marginBottom": "20px",
            },
            children=[
                html.Div(
                    style={"flex": "1"},
                    children=[
                        html.Label(
                            "Metric:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "10px",
                            },
                        ),
                        dcc.RadioItems(
                            id="metric-radio",
                            options=[
                                {"label": i, "value": v}
                                for i, v in [
                                    ("Population", "pop"),
                                    ("Life Expectancy", "lifeExp"),
                                    ("GDP/Cap", "gdpPercap"),
                                ]
                            ],
                            value="lifeExp",
                            inline=True,
                            inputStyle={"marginRight": "5px"},
                            labelStyle={"marginRight": "15px"},
                        ),
                    ],
                ),
                html.Div(
                    style={"width": "200px"},
                    children=[
                        html.Label(
                            "Year:",
                            style={
                                "fontWeight": "bold",
                                "display": "block",
                                "marginBottom": "10px",
                            },
                        ),
                        dcc.Dropdown(
                            id="year-dropdown",
                            options=[{"label": str(y), "value": y} for y in years],
                            value=2007,
                            clearable=False,
                        ),
                    ],
                ),
            ],
        ),
        # Main Content (Graphs)
        html.Main(
            style={"display": "flex", "gap": "20px", "marginBottom": "20px"},
            children=[
                dcc.Graph(
                    id="bar-chart",
                    style={
                        "flex": "1",
                        "backgroundColor": "white",
                        "borderRadius": "8px",
                    },
                ),
                dcc.Graph(
                    id="geo-map",
                    style={
                        "flex": "1",
                        "backgroundColor": "white",
                        "borderRadius": "8px",
                    },
                ),
            ],
        ),
        # Data Table Section
        html.Section(
            style={
                "backgroundColor": "white",
                "padding": "20px",
                "borderRadius": "8px",
            },
            children=[
                html.H3("Raw Data View", style={"marginTop": "0"}),
                dag.AgGrid(
                    id="data-grid",
                    columnDefs=column_defs,
                    defaultColDef={"sortable": True, "filter": True, "resizable": True},
                    dashGridOptions={"pagination": True, "paginationPageSize": 10},
                    style={"height": "400px", "width": "100%"},
                ),
            ],
        ),
    ],
)


# 4. Functional Logic (Callbacks)
@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-dropdown", "value"),
)
def update_app(metric, year):
    dff = df[df["year"] == year]

    # Bar Chart
    fig_bar = px.bar(
        dff.groupby("continent")[metric].mean().reset_index(),
        x="continent",
        y=metric,
        color="continent",
        title=f"Mean {metric} by Continent",
        template="plotly_white",
    )

    # Geo Map
    fig_map = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metric,
        color="continent",
        title=f"Global {metric} in {year}",
        projection="natural earth",
        template="plotly_white",
    )

    return fig_bar, fig_map, dff.to_dict("records")


if __name__ == "__main__":
    app.run(debug=True)
```

### Dash with Dash Bootstrap Components (free)

```python
import dash_ag_grid as dag
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. CARGA DE DATOS
df = pd.read_csv("resources/gapminder.csv")
lista_años = sorted(df["year"].unique())

# 2. CONFIGURACIÓN DE COLUMNAS AG GRID (Con Truco de Formateo)
column_defs = [
    {"field": "country", "headerName": "País"},
    {"field": "year", "headerName": "Año"},
    {"field": "continent", "headerName": "Continente"},
    {
        "field": "pop",
        "headerName": "Población",
        "valueFormatter": {"function": "d3.format(',.0f')(params.value)"},
    },
    {
        "field": "lifeExp",
        "headerName": "Expectativa de Vida",
        "valueFormatter": {"function": "d3.format('.1f')(params.value)"},
    },
    {
        "field": "gdpPercap",
        "headerName": "PIB per Cápita",
        "valueFormatter": {"function": "d3.format('$,.2f')(params.value)"},
    },
]

# 3. INICIALIZAR APP CON TEMA BOOTSTRAP (FLATLY)
app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# --- ABSTRACCIÓN DE COMPONENTES (Organización del código) ---

# Selector de Métrica
selector_metrica = dbc.Col(
    [
        html.Label("Selecciona Métrica:", className="fw-bold"),
        dbc.RadioItems(
            id="metric-radio",
            options=[
                {"label": "Población", "value": "pop"},
                {"label": "Expectativa de Vida", "value": "lifeExp"},
                {"label": "PIB per Cápita", "value": "gdpPercap"},
            ],
            value="lifeExp",
            inline=True,
        ),
    ],
    width=8,
)

# Selector de Año
selector_año = dbc.Col(
    [
        html.Label("Selecciona Año:", className="fw-bold"),
        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": str(y), "value": y} for y in lista_años],
            value=2007,
            clearable=False,
        ),
    ],
    width=4,
)

# Tarjeta de Control Combinada
tarjeta_controles = dbc.Row(
    [
        dbc.Col(
            [
                dbc.Card(
                    dbc.CardBody(dbc.Row([selector_metrica, selector_año])),
                    className="shadow-sm mb-4 border-0",
                )
            ]
        )
    ]
)

# --- DISEÑO PRINCIPAL (LAYOUT) ---

app.layout = dbc.Container(
    [
        # Fila de Encabezado
        dbc.Row(
            [
                dbc.Col(
                    html.H1(
                        "🌍 Explorador Gapminder Pro",
                        className="text-center my-4 text-primary fw-bold",
                    )
                )
            ]
        ),
        # Panel de Controles (Variable definida arriba)
        tarjeta_controles,
        # Fila de Gráficos
        dbc.Row(
            [
                dbc.Col(dcc.Graph(id="bar-chart"), lg=6, md=12),
                dbc.Col(dcc.Graph(id="geo-map"), lg=6, md=12),
            ],
            className="mb-4",
        ),
        # Fila de Tabla
        dbc.Row(
            [
                dbc.Col(
                    [
                        html.H3(
                            "📋 Desglose de Datos", className="mb-3 text-secondary"
                        ),
                        dag.AgGrid(
                            id="data-grid",
                            columnDefs=column_defs,
                            defaultColDef={
                                "sortable": True,
                                "filter": True,
                                "resizable": True,
                            },
                            dashGridOptions={
                                "pagination": True,
                                "paginationPageSize": 10,
                            },
                            style={"height": "400px", "width": "100%"},
                            className="ag-theme-alpine shadow-sm",
                        ),
                    ]
                )
            ],
            className="pb-5",
        ),
    ],
    fluid=True,
    style={"backgroundColor": "#f8f9fa", "minHeight": "100vh"},
)

# --- LÓGICA DE FUNCIONAMIENTO (CALLBACK) ---


@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-dropdown", "value"),
)
def actualizar_dashboard(metrica, año):
    dff = df[df["year"] == año]

    # Diccionario para etiquetas limpias
    nombres = {
        "pop": "Población",
        "lifeExp": "Expectativa de Vida",
        "gdpPercap": "PIB per Cápita",
    }

    # 1. Gráfico de Barras con TRUCO de Transición
    fig_bar = px.histogram(
        dff,
        x="continent",
        y=metrica,
        histfunc="avg",
        color="continent",
        title=f"Promedio de {nombres[metrica]} por Continente ({año})",
        labels={"continent": "Continente", metrica: nombres[metrica]},
        template="plotly_white",
    )
    fig_bar.update_layout(transition_duration=500)  # TRUCO: Animación suave

    # 2. Mapa con TRUCO de Hover Limpio
    fig_map = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metrica,
        color="continent",
        hover_name="country",
        title=f"Distribución Global: {nombres[metrica]} ({año})",
        projection="natural earth",
        template="plotly_white",
        # TRUCO: Solo mostrar lo necesario en el tooltip
        hover_data={metrica: ":,.2f", "continent": False, "year": False},
    )
    fig_map.update_layout(transition_duration=500)

    return fig_bar, fig_map, dff.to_dict("records")


if __name__ == "__main__":
    app.run(debug=True)

```

### PRO

```python
import dash_ag_grid as dag
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, callback, dcc, html

# 1. DATOS
df = pd.read_csv("resources/gapminder.csv")
lista_años = sorted(df["year"].unique())

# TRUCO A: Estilo Condicional en AG Grid (Resaltar valores bajos)
# Si el PIB es menor a 2000, el texto será rojo y negrita.
column_defs = [
    {"field": "country", "headerName": "País"},
    {"field": "continent", "headerName": "Continente"},
    {
        "field": "gdpPercap",
        "headerName": "PIB p.C.",
        "valueFormatter": {"function": "d3.format('$,.0f')(params.value)"},
        "cellStyle": {
            "styleConditions": [
                {
                    "condition": "params.value < 2000",
                    "style": {"color": "#e74c3c", "fontWeight": "bold"},
                },
            ]
        },
    },
    {
        "field": "lifeExp",
        "headerName": "Vida",
        "valueFormatter": {"function": "d3.format('.1f')(params.value)"},
    },
]

app = Dash(
    __name__, external_stylesheets=[dbc.themes.LUX]
)  # Cambiamos a LUX para un look más "Premium"

# --- INTERFAZ ---
app.layout = dbc.Container(
    [
        dbc.Row(
            dbc.Col(
                html.H1(
                    "📊 Business Intelligence: Gapminder", className="text-center my-4"
                )
            )
        ),
        dbc.Row(
            [
                dbc.Col(
                    [
                        html.Label("Métrica:", className="fw-bold"),
                        dbc.Select(
                            id="metric-radio",
                            options=[
                                {"label": "Población", "value": "pop"},
                                {"label": "PIB", "value": "gdpPercap"},
                            ],
                            value="gdpPercap",
                        ),
                    ],
                    width=4,
                ),
                dbc.Col(
                    [
                        html.Label("Año:", className="fw-bold"),
                        dcc.Slider(
                            min(lista_años),
                            max(lista_años),
                            step=None,
                            marks={str(y): str(y) for y in lista_años},
                            value=2007,
                            id="year-slider",
                        ),
                    ],
                    width=8,
                ),
            ],
            className="mb-4 p-3 bg-white shadow-sm rounded",
        ),
        # TRUCO B: dcc.Loading (El "Spinner")
        # Esto envuelve los gráficos. Mientras el callback calcula, aparece un cargador profesional.
        dbc.Row(
            [
                dbc.Col(
                    dcc.Loading(
                        dcc.Graph(id="bar-chart"), type="circle", color="#2c3e50"
                    ),
                    lg=6,
                ),
                dbc.Col(
                    dcc.Loading(dcc.Graph(id="geo-map"), type="cube", color="#2c3e50"),
                    lg=6,
                ),
            ]
        ),
        dbc.Row(
            [
                dbc.Col(
                    dag.AgGrid(
                        id="data-grid",
                        columnDefs=column_defs,
                        dashGridOptions={"pagination": True},
                        style={"height": "400px", "width": "100%"},
                        className="ag-theme-alpine mt-4 shadow",
                    )
                )
            ]
        ),
    ],
    fluid=True,
    style={"backgroundColor": "#f4f7f6"},
)


# --- LÓGICA ---
@callback(
    Output("bar-chart", "figure"),
    Output("geo-map", "figure"),
    Output("data-grid", "rowData"),
    Input("metric-radio", "value"),
    Input("year-slider", "value"),
)
def update(metrica, año):
    dff = df[df["year"] == año]

    # Gráfico de barras
    fig_bar = px.bar(
        dff.nlargest(10, metrica),
        x="country",
        y=metrica,
        color="country",
        template="plotly_white",
        title=f"Top 10 Países ({año})",
    )
    fig_bar.update_layout(showlegend=False, transition_duration=800)

    # Mapa con TRUCO C: Hover Template Personalizado (HTML en el tooltip)
    fig_map = px.scatter_geo(
        dff,
        locations="country",
        locationmode="country names",
        size=metrica,
        color="continent",
        template="plotly_white",
    )

    fig_map.update_traces(
        hovertemplate="<b>País:</b> %{location}<br><b>Valor:</b> %{marker.size:,}<extra></extra>"
    )
    fig_map.update_layout(transition_duration=800)

    return fig_bar, fig_map, dff.to_dict("records")


if __name__ == "__main__":
    app.run(debug=True)

```